In [1]:
import pandas as pd

df = pd.read_csv("./tox21.csv.gz")

print(df.shape)
print(df.columns)

(8014, 14)
Index(['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD',
       'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53',
       'mol_id', 'smiles'],
      dtype='str')


In [6]:
print(df[['smiles', 'NR-AhR']].head())
print("\n")
df[['smiles', 'NR-AhR']].count()

                                              smiles  NR-AhR
0                       CCOc1ccc2nc(S(N)(=O)=O)sc2c1     1.0
1                          CCN1C(=O)NC(c2ccccc2)C1=O     0.0
2  CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...     NaN
3                    CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C     0.0
4                          CC(O)(P(=O)(O)O)P(=O)(O)O     NaN




smiles    8014
NR-AhR    6691
dtype: int64

In [7]:
data = df[['smiles', 'NR-AhR']].copy()
data = data.dropna(subset=['NR-AhR']).copy()

print(data.shape)
print(data['NR-AhR'].value_counts())

(6691, 2)
NR-AhR
0.0    5906
1.0     785
Name: count, dtype: int64


In [ ]:
%pip install rdkit



   ---------------------------------------- 0.0/25.5 MB ? eta -:--:--
   - -------------------------------------- 0.8/25.5 MB 7.3 MB/s eta 0:00:04
   - -------------------------------------- 1.0/25.5 MB 7.2 MB/s eta 0:00:04
   --- ------------------------------------ 2.1/25.5 MB 4.5 MB/s eta 0:00:06
   ---- ----------------------------------- 3.1/25.5 MB 4.0 MB/s eta 0:00:06
   ------- -------------------------------- 4.7/25.5 MB 4.8 MB/s eta 0:00:05
   --------- ------------------------------ 6.0/25.5 MB 5.2 MB/s eta 0:00:04
   ----------- ---------------------------- 7.6/25.5 MB 5.5 MB/s eta 0:00:04
   ------------- -------------------------- 8.4/25.5 MB 5.6 MB/s eta 0:00:04
   -------------- ------------------------- 9.4/25.5 MB 5.4 MB/s eta 0:00:03
   ---------------- ----------------------- 10.2/25.5 MB 5.2 MB/s eta 0:00:03
   ---------------- ----------------------- 10.7/25.5 MB 4.8 MB/s eta 0:00:04
   ------------------ --------------------- 11.5/25.5 MB 4.9 MB/s eta 0:00:03
   


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from rdkit import Chem

In [11]:
mol = Chem.MolFromSmiles(data['smiles'].iloc[0])

print(mol)

In [12]:
data['mol'] = data['smiles'].apply(Chem.MolFromSmiles)

[00:00:52] WARNING: not removing hydrogen atom without neighbors
[00:00:53] Explicit valence for atom # 3 Al, 6, is greater than permitted
[00:00:53] Explicit valence for atom # 4 Al, 6, is greater than permitted
[00:00:53] Explicit valence for atom # 4 Al, 6, is greater than permitted
[00:00:53] Explicit valence for atom # 9 Al, 6, is greater than permitted
[00:00:53] Explicit valence for atom # 5 Al, 6, is greater than permitted
[00:00:53] Explicit valence for atom # 16 Al, 6, is greater than permitted
[00:00:53] Explicit valence for atom # 20 Al, 6, is greater than permitted


In [13]:
print(data['mol'].isna().sum())

7


In [14]:
invalid = data[data['mol'].isna()]

print(invalid[['smiles', 'NR-AhR']])

                                                 smiles  NR-AhR
2308                               O=CO[AlH3](OC=O)OC=O     0.0
2315                                  CC(=O)O[AlH3](O)O     0.0
3599                            CC(=O)O[AlH3](O)OC(C)=O     0.0
4632            CCOC(=O)/C=C(/C)O[AlH3](OC(C)CC)OC(C)CC     0.0
4716                            CCCCO[AlH3](OCCCC)OCCCC     0.0
5627  O=S(=O)(OC[C@H]1O[C@H](O[C@]2(COS(=O)(=O)O[AlH...     0.0
6859                  CCCCCCCCCCCCCCCCCC(=O)O[AlH3](O)O     0.0


In [15]:
print("Invalid molecules:", len(invalid))

Invalid molecules: 7


In [16]:
data = data[data['mol'].notna()].copy()

In [17]:
print(data.shape)

(6684, 3)


In [18]:
data['canonical_smiles'] = data['mol'].apply(
    Chem.MolToSmiles
)

In [19]:
print(data[['smiles', 'canonical_smiles']].head())

                                 smiles                      canonical_smiles
0          CCOc1ccc2nc(S(N)(=O)=O)sc2c1          CCOc1ccc2nc(S(N)(=O)=O)sc2c1
1             CCN1C(=O)NC(c2ccccc2)C1=O             CCN1C(=O)NC(c2ccccc2)C1=O
3       CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C       CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C
5  CC(C)(C)OOC(C)(C)CCC(C)(C)OOC(C)(C)C  CC(C)(C)OOC(C)(C)CCC(C)(C)OOC(C)(C)C
6                   O=S(=O)(Cl)c1ccccc1                   O=S(=O)(Cl)c1ccccc1


In [20]:
print(data['canonical_smiles'].duplicated().sum())

142


In [21]:
duplicate_rows = data[
    data['canonical_smiles'].duplicated(keep=False)
]

print(duplicate_rows[
    ['canonical_smiles', 'NR-AhR']
].sort_values('canonical_smiles'))

                                       canonical_smiles  NR-AhR
5823         C/C(=C(/CCO)SSCC1CCCO1)N(C=O)Cc1cnc(C)nc1N     0.0
827          C/C(=C(/CCO)SSCC1CCCO1)N(C=O)Cc1cnc(C)nc1N     0.0
1495                                           C1CNCCN1     0.0
6380                                           C1CNCCN1     0.0
49    C=C[C@H]1CN2CCC1C[C@H]2[C@H](O)c1ccnc2ccc(OC)cc12     1.0
...                                                 ...     ...
3734                          c1ccc2cc(CC3=NCCN3)ccc2c1     0.0
3335                                           c1ccncc1     0.0
7948                                           c1ccncc1     0.0
5409                             c1cnc2c(c1)ccc1cccnc12     1.0
4848                             c1cnc2c(c1)ccc1cccnc12     0.0

[266 rows x 2 columns]


In [22]:
conflicting = (
    data.groupby('canonical_smiles')['NR-AhR']
    .nunique()
)

conflicting = conflicting[conflicting > 1]

print("Conflicting structures:", len(conflicting))

Conflicting structures: 8


In [23]:
print(
    data[data['canonical_smiles'].isin(conflicting.index)]
    [['canonical_smiles', 'NR-AhR']]
    .sort_values('canonical_smiles')
)

               canonical_smiles  NR-AhR
5707  CC(C)NCC(O)c1ccc(O)c(O)c1     0.0
2291  CC(C)NCC(O)c1ccc(O)c(O)c1     1.0
7639         CCCC[n+]1ccc(C)cc1     1.0
2404         CCCC[n+]1ccc(C)cc1     0.0
3626         CCCC[n+]1ccc(C)cc1     0.0
1363            Cc1ccc(N)c(C)c1     1.0
7279            Cc1ccc(N)c(C)c1     0.0
6669               Cc1cccc(N)c1     1.0
5318               Cc1cccc(N)c1     0.0
660     Nc1ccc(-c2ccc(N)cc2)cc1     0.0
665     Nc1ccc(-c2ccc(N)cc2)cc1     1.0
1986                  Nc1ccccc1     0.0
1985                  Nc1ccccc1     1.0
7105  c1ccc2cc(CC3=NCCN3)ccc2c1     1.0
3734  c1ccc2cc(CC3=NCCN3)ccc2c1     0.0
5409     c1cnc2c(c1)ccc1cccnc12     1.0
4848     c1cnc2c(c1)ccc1cccnc12     0.0


In [24]:
data = data[
    ~data['canonical_smiles'].isin(conflicting.index)
].copy()

In [25]:
print(data.shape)

(6667, 4)


In [26]:
data = data.drop_duplicates(
    subset='canonical_smiles',
    keep='first'
).copy()

In [27]:
print(data.shape)

(6534, 4)


In [28]:
print(data['NR-AhR'].value_counts())

NR-AhR
0.0    5774
1.0     760
Name: count, dtype: int64


In [29]:
clean_data = data[['smiles', 'NR-AhR']].copy()

clean_data.to_csv(
    'tox21_nr_ahr_clean.csv',
    index=False
)